In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableWithMessageHistory
from dotenv import load_dotenv 
import langchain 
import langchain_community
from langchain_openai import ChatOpenAI
import os

/tmp/ipykernel_53847/2153619476.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


In [9]:
from langchain_ollama.chat_models import ChatOllama 

model = ChatOllama(model = "gemma4:31b-cloud", temperature= 0.7)
message = [{"role" :"system" ,"content" :  "you are the helpfull assistant" , "role" : "user" , "content" : "hi" }]
model.invoke(message)

# help(langchain_ollama.chat_models)

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'model': 'gemma4:31b', 'created_at': '2026-07-19T12:35:28.832064549Z', 'done': True, 'done_reason': 'stop', 'total_duration': 293019515, 'load_duration': None, 'prompt_eval_count': 14, 'prompt_eval_duration': None, 'eval_count': 10, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b', 'model_provider': 'ollama'}, id='lc_run--019f7a5f-d30e-78f0-a368-696d713599c2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 10, 'total_tokens': 24})

In [10]:
load_dotenv()

model_name = "gemma4:31b-cloud"

store = {}
def chat_history(session_id : str) :
    if session_id not in store :
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]
def chatmodel() :
    model = ChatOllama(model = model_name, temperature= 0.7)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system" , "your the helpfull assistant"),
        ("placeholder" , "{history}"),
        ("human" , "{user_query}") 
    ]
    )
    chain= prompt | model | StrOutputParser()
    
    chain_with_memory = RunnableWithMessageHistory(
        chain , 
        chat_history,
        input_messages_key="user_query",
        history_messages_key="history"
    )
    
    return chain_with_memory

In [11]:
chat = chatmodel()
response_1 = chat.invoke({"user_query" : "hi"} , {"configurable" : {"session_id" : "test-1"}})
response_1

/tmp/ipykernel_53847/613075644.py:1: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  chat = chatmodel()


'Hello! How can I help you today?'

In [12]:
response_2 = chat.invoke({"user_query" : "my name is praveen"} , {"configurable" : {"session_id" : "test-1"}})
response_2

"It's nice to meet you, Praveen! How is your day going so far? Is there anything I can help you with?"

In [13]:
response_3 = chat.invoke({"user_query" : "what is my name , and also tell the what we talk in previous chat"} , {"configurable" : {"session_id" : "test-1"}})
response_3

'Your name is **Praveen**.\n\nAs for our previous conversation, we just started! You said hello, I greeted you back, and then you introduced yourself by telling me your name.'

In [14]:
response_4 = chat.invoke({"user_query" : "hi, what is my name"} , {"configurable" : {"session_id" : "test-2"}})
response_4

"I don't know your name yet! Since we are chatting for the first time (or I don't have access to your personal profile), you haven't told me what it is.\n\n**What is your name?** I'd love to know!"

In [15]:
from langchain_core.messages import HumanMessage , AIMessage

model_name = "openai/gpt-oss-20b:free"

api = os.getenv("open-api")

model = ChatOpenAI(model = model_name, temperature= 0.7 , api_key= api , base_url = "https://openrouter.ai/api/v1")

conversation =[]
def without_memory(user_query :str):
    conversation.append(HumanMessage(content=user_query))
    chat = model.invoke(conversation)
    conversation.append(AIMessage(content=chat.content))
    return chat.content


response_11 = without_memory("hi , i am praveen")
response_12 = without_memory("what is my name , and also tell the previouse question")
response_12


'Your name is **Praveen**.  \nThe previous question you asked was: *“hi , i am praveen.”*'

In [20]:
roles = {
    "1": {
        "name": "Python Tutor",
        "prompt": "You are a patient and friendly Python tutor. Explain concepts clearly with examples."
    },
    "2": {
        "name": "Fitness Coach",
        "prompt": "You are a practical fitness coach. Give simple, actionable health and exercise advice."
    },
    "3": {
        "name": "Travel Guide",
        "prompt": "You are a knowledgeable travel guide. Suggest places, tips, and cultural insights."
    }
}


def choose_role():
    while True : 
        for key , role in roles.items():
            print(key , role['name'])
        chooise = input("Enter your chooise").strip()
    
        if chooise in roles : 
            role = roles[chooise]
            print(role['name'])
            return role 
        else :
            print("invalid chooise , choose the avalivable role")
            
model = ChatOllama(model = "gemma4:31b-cloud" , temperature= 0.7) 
        
def chatloop():
    while True :
        role = choose_role()
        message = [("system" , role['prompt'])]
        
        
        while True : 
            
            user_input = input("enter : ").strip()
            
            if user_input.lower() in ("exit", "quit") :
                print("returning")
                return
            
            if user_input.lower() == "switch":
                print("switch role")
                break
            
            message.append(("user" ,user_input))
            
            res = model.invoke(message)
            
            message.append(AIMessage(content=res.content))
            
                 
            print(res.content)

chatloop()
    

1 Python Tutor
2 Fitness Coach
3 Travel Guide
Python Tutor
Hello! I'd be happy to explain decorators to you. 

At first glance, decorators can seem a bit like "magic," but once you understand how functions work in Python, they become very logical.

### What is a Decorator?

In simple terms, a **decorator** is a function that takes another function and extends its behavior without explicitly modifying its code. 

Think of it like adding a "wrapper" around a gift. The gift inside stays the same, but the wrapping paper adds a new look or extra flair.

---

### A Simple Example: The "Greeting" Decorator

Let's say you have a function that prints a message. You want to add a "Welcome" and "Goodbye" message every time that function runs, but you don't want to type those messages inside every single function you write.

Here is how you do it:

```python
# 1. This is the Decorator function
def my_decorator(func):
    def wrapper():
        print("--- Welcome! The function is starting ---") 
  

In [21]:

def sum(a , b= 10):
    return a * b + 1

print(sum(10))

101


In [ ]:
a = lambda x, b=10 : x* b
a(5)

50

In [29]:
# def text(string):
#     return string.upper()

# text("hi")

text = lambda text : text.upper()

text("hello")

'HELLO'